In [0]:
# Notebook: Setup_Existing_Catalog

# Fix: Define choices first to ensure sync
all_layers = ["bronze", "silver", "gold", "security"]

dbutils.widgets.text("project_catalog", "vstone_catalog", "1. Target Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema Name")

# Fix: Default value (2nd param) must be part of choices (3rd param)
dbutils.widgets.multiselect("schemas", "bronze", all_layers, "3. Target Schemas")

# Fetching variables
CATALOG = dbutils.widgets.get("project_catalog")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
# Converting comma-separated string from widget to list
SELECTED_SCHEMAS = dbutils.widgets.get("schemas").split(",")

In [0]:
# 1. Use existing Catalog (Avoiding CREATE to prevent PERMISSION_DENIED)
try:
    spark.sql(f"USE CATALOG {CATALOG}")
    print(f"Successfully pointed to catalog: {CATALOG}")
except Exception as e:
    print(f"Error: Catalog {CATALOG} not found. Ensure it exists in Metastore.")
    raise e

# 2. Schema Verification (Using existing structure from Screenshot)
# We only create if missing, otherwise we use existing
for schema_name in SELECTED_SCHEMAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}")
    print(f"Schema verified: {CATALOG}.{schema_name}")

# 3. Volume Verification (Using RAW_SCHEMA variable)
# Hardcoding removed from landing and chunks paths
volumes = ["landing", "chunks"]

for vol in volumes:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}.{vol}")
    print(f"Volume ready: /Volumes/{CATALOG}/{RAW_SCHEMA}/{vol}")

print(f"""
 READY FOR PIPELINE RESTART
--------------------------------------------------
Catalog: {CATALOG}
Raw Schema (Landing): {RAW_SCHEMA}
Active Layers: {', '.join(SELECTED_SCHEMAS)}
--------------------------------------------------
""")